# Module 35 — Exercise 2: Token Bucket Rate Limiter

The Token Bucket algorithm regulates traffic rates, accommodating temporary bursts while strictly enforcing sustained throughput limits.

In this exercise, you will implement a thread-safe / monotonic Token Bucket Rate Limiter in Python.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 35 README |



# Your turn


### Task 1: Implement `TokenBucketRateLimiter`

Implement `TokenBucket(rate_per_sec, capacity)`:
- Tokens refill continuously at `rate_per_sec` up to `capacity`.
- `allow(tokens=1)`: If at least `tokens` are available, deduct and return `True`. Otherwise return `False`.


In [ ]:
# ANSWER 1
import time

class TokenBucket:
    def __init__(self, rate_per_sec: float, capacity: float):
        self.rate = rate_per_sec
        self.capacity = capacity
        self.tokens = capacity
        self.last_refill = time.monotonic()

    def _refill(self) -> None:
        now = time.monotonic()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_refill = now

    def allow(self, tokens: float = 1.0) -> bool:
        self._refill()
        if self.tokens >= tokens:
            self.tokens -= tokens
            return True
        return False



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

# Capacity 3, refill rate 1 token / second
tb = TokenBucket(rate_per_sec=10.0, capacity=3.0)

# Consume 3 burst tokens
b1 = tb.allow(1.0)
b2 = tb.allow(1.0)
b3 = tb.allow(1.0)
# 4th immediate token should be rejected
b4 = tb.allow(1.0)

time.sleep(0.12)  # Wait for ~1.2 tokens to refill
b5 = tb.allow(1.0)

results = [
    check(b1 and b2 and b3, "Task 1: Initial burst of 3 tokens allowed"),
    check(b4 is False, "Task 1: Immediate 4th token rejected when bucket was empty"),
    check(b5 is True, "Task 1: Token allowed after refill delay"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

